In [7]:
import os
import datetime

import IPython
import IPython.display
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.graphics.gofplots import qqplot
from statsmodels.tsa.stattools import adfuller

from tqdm import tqdm_notebook
from itertools import product
from typing import Union

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

mpl.rcParams['figure.figsize'] = (8, 6)
mpl.rcParams['axes.grid'] = False

In [8]:
path = "../../datasets/TxSON_data_2026-02-24/"
stations_list = ['CB01', 'CB04', 'CB06', 'CB07', 'CB09', 'CB10']

dfs = {}
for code in stations_list:
    df = pd.read_csv(path + code + '.dat', skiprows=5, parse_dates=['Date'], index_col='Date')
    df.drop(['Flag'], axis=1, inplace=True)
    dfs[code] = df

In [9]:
# Drop SWC_50 and T_50 (CB07 lacks them, errors='ignore' handles that uniformly)
for key in dfs.keys():
    dfs[key].drop(['SWC_50', 'T_50'], axis=1, errors='ignore', inplace=True)

In [10]:
# TxSON files have a single Ppt column and no NaN rows to drop after Flag removal
for station, df in dfs.items():
    df.dropna(inplace=True)
    dfs[station] = df

In [11]:
# TxSON soil files do not contain wind data — wind vectorization skipped

In [12]:
# Remove periodicity in time data (remove daily and yearly periodicity)
day = 24*60*60
year = (365.2425)*day

for station, df in dfs.items() :
  timestamp_s = (df.index).map(pd.Timestamp.timestamp)

  df['Day sin'] = np.sin(timestamp_s * (2 * np.pi / day))
  df['Day cos'] = np.cos(timestamp_s * (2 * np.pi / day))
  df['Year sin'] = np.sin(timestamp_s * (2 * np.pi / year))
  df['Year cos'] = np.cos(timestamp_s * (2 * np.pi / year))

  dfs[station] = df

In [13]:
# TxSON has a single Ppt column — no Ppt_soil/Ppt_met merge needed

In [14]:
#Normalize all the data

for key in dfs.keys():
  dfs[key] = (dfs[key] - dfs[key].min())/(dfs[key].max()-dfs[key].min())

In [15]:
# Geographic coordinates are not included in TxSON .dat files — position features skipped

In [16]:
#only use data shared all together: indexes that are none null for each station

index_union = pd.Index([])
for station, df in dfs.items():
  index_union = index_union.union(df.index)

index_int = index_union
for station, df in dfs.items():
  index_int = index_int.intersection(df.index)


print(len(index_int))


64496


# Create Multi-Index DF for all Station Data

In [17]:
feats = dfs['CB01'].columns.tolist()

stations = list(dfs.keys())

In [18]:
index = index_int

cols = pd.MultiIndex.from_product([stations,feats], names = ['Station', 'Feature'])

In [19]:
for station, df in dfs.items():
  cols_new = []
  for col in df.columns.tolist():
    cols_new.append(f"{station} {col}")
  df.columns = cols_new

In [20]:
lis = [dfs[key].loc[index] for key in dfs.keys()]

data = pd.concat(lis, axis = 1)

data.head()

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [ ]:
df = pd.DataFrame(data = data.values, index = index, columns = cols)

# Data Split

In [ ]:
#create Df of our target Values

target_station = 'CB07'

target_names = ['SWC_5']

target_df = df[target_station][target_names]

target_df.shape

In [ ]:
#Create DF of all our data not the target

non_targets = list(dfs.keys())

non_targets.remove('CB07')

train_df = df[non_targets]

train_df.shape

train=[]
test=[]

for station, feature in df:
    if feature == "SWC_5" and station != 'CB07':
        train.extend(df[station].SWC_5)
    elif feature == "SWC_5" and station == 'CB07':
        test.extend(df[station].SWC_5)

In [ ]:
df_train = pd.DataFrame(train[::24*30])
df_test = pd.DataFrame(test[::24*30])
df_total = pd.concat([df_train, df_test], ignore_index=True)
# df_Tair_exog = pd.DataFrame(Tair_exog[::24*30])

train_len = len(df_train)
test_len = len(df_test)

total_len = train_len + test_len

# want them seperate, one vector each statement
# train sequentially

# one per day (not month)
# adjust time to be index and use mean()
# youtube vidio under referances in git

In [ ]:
df_test

In [ ]:
# for station, feature in df:
    
#     if feature == "SWC_5":
    
#         fig, ax = plt.subplots()

#         ax.plot(df[station].SWC_5)
#         ax.set_xlabel('Date')
#         ax.set_ylabel('Soil Water Content')

#         fig.autofmt_xdate()
#         plt.tight_layout()

In [ ]:
# Define SARIMA model
from typing import Union
from tqdm import tqdm_notebook
from statsmodels.tsa.statespace.sarimax import SARIMAX

def optimize_SARIMAX(endog: Union[pd.Series, list], exog: Union[pd.Series, list], order_list: list, d: int, D: int, s: int) -> pd.DataFrame:
    
    results = []
    
    for order in tqdm_notebook(order_list):
        try: 
            model = SARIMAX(
                endog,
                exog,
                order=(order[0], d, order[1]),
                seasonal_order=(order[2], D, order[3], s),
                simple_differencing=False).fit(disp=False)
        except:
            continue
            
        aic = model.aic
        print([order, aic])
        results.append([order, model.aic])
        
    result_df = pd.DataFrame(results)
    result_df.columns = ['(p,q,P,Q)', 'AIC']
    
    #Sort in ascending order, lower AIC is better
    result_df = result_df.sort_values(by='AIC', ascending=True).reset_index(drop=True)
    
    return result_df

In [ ]:
# Define range of parameter's to check
ps = range(0, 2, 1)
qs = range(0, 2, 1)
Ps = range(0, 2, 1)
Qs = range(0, 2, 1)

order_list = list(product(ps, qs, Ps, Qs))

d = 0
D = 0
s = 12


In [ ]:
# Find best set of parameters using the AIC
SARIMA_result_df = optimize_SARIMAX(df_train, None, order_list, d, D, s)
SARIMA_result_df

In [ ]:
# Define and fit SARIMAX model
# call on each or optimize each
# N X 5 station to train (.value turns into a numpy array)
SARIMA_model = SARIMAX(df_train, order=((SARIMA_result_df.iloc[0][0][0]), 0, SARIMA_result_df.iloc[0][0][1]), seasonal_order = (SARIMA_result_df.iloc[0][0][2], 0, SARIMA_result_df.iloc[0][0][3], s), simple_differencing=False, enforce_stationarity=False)
SARIMA_model_fit = SARIMA_model.fit(disp=False)

print(SARIMA_model_fit.summary())

In [ ]:
## Forecasting 

def rolling_forecast(df: pd.DataFrame, train_len: int, horizon: int, window: int, method: str) -> list:
    
    total_len = train_len + horizon
    
    if method == 'last_season':
        pred_last_season = []
            
        for i in range(train_len, total_len, window):
            last_season = df[0][i-window:i]
            pred_last_season.extend(last_season)
            
        return pred_last_season
    
    elif method == 'SARIMA':
        pred_SARIMA = []
        
        for i in range(train_len, total_len, window):
            model = SARIMAX(df[:i], order=(SARIMA_result_df.iloc[0][0][0], 0, SARIMA_result_df.iloc[0][0][1]), seasonal_order = (SARIMA_result_df.iloc[0][0][2], 0, SARIMA_result_df.iloc[0][0][3], s), simple_differencing=False)
            res = model.fit(disp=False)
            predictions = res.get_prediction(0, i + window - 1)
            oos_pred = predictions.predicted_mean[-window:]
            pred_SARIMA.extend(oos_pred)
            
        return pred_SARIMA

In [ ]:
pred_df = df_test.copy()

In [ ]:
# Define rolling forcast structure and create baseline for model by recalling the previous season
TRAIN_LEN = train_len
HORIZON = test_len
WINDOW = test_len

pred_df['last_season'] = rolling_forecast(df_train, TRAIN_LEN, HORIZON, WINDOW, 'last_season')
pred_df.last_season

In [ ]:
# Define SARIMA Prediction Dataframe
pred_df['SARIMA'] = rolling_forecast(df_total, TRAIN_LEN, HORIZON, WINDOW, 'SARIMA')

pred_df.SARIMA

In [ ]:
# visualize predictions
fig, ax = plt.subplots()

# ax.plot(df_total)
ax.plot(df_test, 'b-', label='actual')
# ax.plot(pred_df.last_season, 'r:', label='naive seasonal')
ax.plot(pred_df.SARIMA, 'k--', label='SARIMA')
ax.set_xlabel('Months')
ax.set_ylabel('Normalized SWC_5')

fig.autofmt_xdate()
plt.tight_layout()

In [ ]:
# Evaluate 

mse=np.mean((df_test-pred_df.SARIMA)**2)
mse